In [ ]:
!pip install roboflow ultralytics python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()  # reads .env from repo root

def get_secret(key):
    val = os.getenv(key)
    if val:
        return val
    # fallback to Colab secrets when .env is absent
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        raise ValueError(f"'{key}' not set in .env or Colab secrets")

In [ ]:
REPO_ROOT = Path(".").resolve()

additional_dataset_directory_path = str(REPO_ROOT / "datasets/additional")
synthetic_dataset_directory_path   = str(REPO_ROOT / "datasets/synthetic")

---

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=get_secret("ROBOFLOW_API_KEY"))
project = rf.workspace(get_secret("ROBOFLOW_WORKSPACE_ID")).project(get_secret("ROBOFLOW_PROJECT_ID"))

dataset = project.version(1).download("yolov11")

In [ ]:
from ultralytics import YOLO
import shutil

# MODEL_OVERRIDE_PATH = None
MODEL_OVERRIDE_PATH = str(REPO_ROOT / "models/bounding_best.pt")

if MODEL_OVERRIDE_PATH:
    model = YOLO(MODEL_OVERRIDE_PATH)
else:
    model = YOLO("yolo11n.pt")  # n/s/m/l/x
    model.train(data=f"{dataset.location}/data.yaml", epochs=10, imgsz=512)

    # model = YOLO("runs/detect/train-2/weights/best.pt")
    # shutil.copy("runs/detect/train-2/weights/best.pt", str(REPO_ROOT / "models/bounding_best.pt"))

In [ ]:
results = model.predict(source=0, show=True, conf=0.25)

---

In [ ]:
TEST_IMAGE = "IMG_0652.JPG"
results = model(additional_dataset_directory_path + "/images/" + TEST_IMAGE)

from IPython.display import display
import PIL.Image
display(PIL.Image.fromarray(results[0].plot()))

In [ ]:
import cv2

output_dir = str(REPO_ROOT / "output")
os.makedirs(output_dir + "/cropped", exist_ok=True)

cv2.imwrite(output_dir + "/" + TEST_IMAGE, results[0].plot())

img = cv2.imread(additional_dataset_directory_path + "/images/" + TEST_IMAGE)

for i, box in enumerate(results[0].boxes.xyxy):
    x1, y1, x2, y2 = map(int, box)
    crop = img[y1:y2, x1:x2]
    cv2.imwrite(f"{output_dir}/cropped/{TEST_IMAGE}-CROP-{i}.jpg", crop)

---